In [1]:
import os

# This forces TensorFlow to use the internal Keras 2 instead of the new Keras 3
os.environ['TF_USE_LEGACY_KERAS'] = '1'

# 07 Milestone Project 1: 🍔👁 Food Vision Big

In the previous notebook we built Food Vision mini: a transfer learning model which beat the original results of the [Food101 paper](https://data.vision.ee.ethz.ch/cvl/datasets_extra/food-101/) with only 10% of the data.

We're going to be building Food Vision Big using all of the data from the Food101 dataset.

All 75,750 training images and 25,250 testing images.

This time **we've got the goal of beating [DeepFood](https://www.researchgate.net/publication/304163308_DeepFood_Deep_Learning-Based_Food_Image_Recognition_for_Computer-Aided_Dietary_Assessment)**, a 2016 paper which used a Convolutional Neural Network trained for 2-3 days to achieve 77.4% top-1 accuracy.

> 🔑 **Note:** **Top-1 accuracy** means "accuracy for the top softmax activation value output by the model" (because softmax ouputs a value for every class, but top-1 means only the highest one is evaluated). **Top-5 accuracy** means "accuracy for the top 5 softmax activation values output by the model", in other words, did the true label appear in the top 5 activation values? Top-5 accuracy scores are usually noticeably higher than top-1.

|  | 🍔👁 Food Vision Big™ | 🍔👁 Food Vision mini |
|-----|-----|-----|
| Dataset source | TensorFlow Datasets | Preprocessed download from Kaggle | 
| Train data | 75,750 images | 7,575 images | 
| Test data | 25,250 images | 25,250 images | 
| Mixed precision | Yes | No |
| Data loading | Performanant tf.data API | TensorFlow pre-built function |  
| Target results | 77.4% top-1 accuracy (beat [DeepFood paper](https://arxiv.org/abs/1606.05675)) | 50.76% top-1 accuracy (beat [Food101 paper](https://data.vision.ee.ethz.ch/cvl/datasets_extra/food-101/static/bossard_eccv14_food-101.pdf)) | 

*Table comparing difference between Food Vision Big (this notebook) versus Food Vision mini (previous notebook).*

Alongside attempting to beat the DeepFood paper, we're going to learn about two methods to significantly improve the speed of our model training:
1. Prefetching
2. Mixed precision training

## What we're going to cover

* Using TensorFlow Datasets to download and explore data
* Creating preprocessing function for our data
* Batching & preparing datasets for modelling (**making our datasets run fast**)
* Creating modelling callbacks
* Setting up **mixed precision training**
* Building a feature extraction model
* Fine-tuning the feature extraction model
* Viewing training results on TensorBoard.

In [2]:
# Getting Helper Functions:
import wget

# Run commands below if the helper_functions.py file is not already downloded:
#url = 'https://raw.githubusercontent.com/Shailya777/Learning-Logs/refs/heads/main/Deep_Learning_with_Tensorflow/helper_functions.py'
#file = wget.download(url)

In [4]:
from helper_functions import create_tensorboard_callback
from helper_functions import compare_historys
from helper_functions import plot_loss_curves

## Use TensorFlow Datasets to Download Data

In previous notebooks, we've downloaded our food images (from the [Food101 dataset](https://www.kaggle.com/dansbecker/food-101/home)) from Google Storage.

And this is a typical workflow you'd use if you're working on your own datasets.

However, there's another way to get datasets ready to use with TensorFlow.

For many of the most popular datasets in the machine learning world (often referred to and used as benchmarks), you can access them through [TensorFlow Datasets (TFDS)](https://www.tensorflow.org/datasets/overview).

What is **TensorFlow Datasets**?

A place for prepared and ready-to-use machine learning datasets.

Why use TensorFlow Datasets?

* Load data already in Tensors
* Practice on well established datasets
* Experiment with differet data loading techniques (like we're going to use in this notebook)
* Experiment with new TensorFlow features quickly (such as mixed precision training)

Why *not* use TensorFlow Datasets?

* The datasets are static (they don't change, like your real-world datasets would)
* Might not be suited for your particular problem (but great for experimenting)

To begin using TensorFlow Datasets we can import it under the alias `tfds`.

In [5]:
# Get Tensorflow Datasets:

# Run command below if Tensorflow Datasets is not already installed:
#!pip install tensorflow_datasets

In [6]:
import tensorflow_datasets as tfds

To find all of the available datasets in TensorFlow Datasets, you can use the `list_builders()` method.

After doing so, we can check to see if the one we're after (`"food101"`) is present.

In [8]:
datasets_list = tfds.list_builders()

In [9]:
print(len(datasets_list))

1314


In [10]:
'food101' in datasets_list

True

In [11]:
datasets_list[:10]

['abstract_reasoning',
 'accentdb',
 'aeslc',
 'aflw2k3d',
 'ag_news_subset',
 'ai2_arc',
 'ai2_arc_with_ir',
 'ai2dcaption',
 'aloha_mobile',
 'amazon_us_reviews']

To get access to the Food101 dataset from the TFDS, we can use the [`tfds.load()`](https://www.tensorflow.org/datasets/api_docs/python/tfds/load) method.

In particular, we'll have to pass it a few parameters to let it know what we're after:
* `name` (str) : the target dataset (e.g. `"food101"`)
* `split` (list, optional) : what splits of the dataset we're after (e.g. `["train", "validation"]`)
  * the `split` parameter is quite tricky. See [the documentation for more](https://github.com/tensorflow/datasets/blob/master/docs/splits.md).
* `shuffle_files` (bool) : whether or not to shuffle the files on download, defaults to `False` 
* `as_supervised` (bool) : `True` to download data samples in tuple format (`(data, label)`) or `False` for dictionary format 
* `with_info` (bool) : `True` to download dataset metadata (labels, number of samples, etc)

> 🔑 **Note:** Calling the `tfds.load()` method will start to download a target dataset to disk if the `download=True` parameter is set (default). This dataset could be 100GB+, so make sure you have space.

`Problem: Downloading large datasets (like Food101, ~5GB) via tfds on a local machine can be extremely slow due to server limits (e.g., 5+ hours). Solution: Use Google Colab's high-speed network to download and process the data, then transfer the finished files to the local machine.`

#### Step 1: (In Google Colab)
Run these commands in a Colab Notebook (Hosted Runtime) to download and prepare the data using Google's fiber internet.

##### 1. Download & Process Data (Takes ~3-5 mins in Colab):

`import tensorflow_datasets as tfds`

`dataset = tfds.load(name= "food101", split= ["train","validation"], as_supervised= True, shuffle_files= True, with_info= True)`

##### 2. Zip the processed data:

Note: TFDS stores data in /root/tensorflow_datasets/ by default in Colab

`!zip -r food101_processed.zip /root/tensorflow_datasets/food101`

##### 3. Move to Google Drive for fast download:

`from google.colab import drive`

`drive.mount('/content/drive')`

`!cp food101_processed.zip /content/drive/MyDrive/`

#### Step 2: Transfer to Local Machine:

1. Download food101_processed.zip from Google Drive to your Windows Downloads folder.

2. Open WSL Terminal and move it to your home directory:

`cp /mnt/c/Users/YOUR_WINDOWS_USERNAME/Downloads/food101_processed.zip ~`

#### Step 3: Extract & Fix Folder Structure (In WSL):
Colab zips the absolute path (/root/...), creating a deep nested structure when unzipped. We must fix this so tfds can find the data.

Run these commands in your WSL Terminal:

##### 1. Create the target directory

`mkdir -p ~/tensorflow_datasets/`

##### 2. Unzip the file:

`unzip -q food101_processed.zip -d ~/tensorflow_datasets/`

##### 3. Fix the Nesting Issue (Moves data from ~/tensorflow_datasets/root/tensorflow_datasets/food101 -> ~/tensorflow_datasets/food101):

`cd ~/tensorflow_datasets`

`mv root/tensorflow_datasets/food101 .`

`rm -rf root  # Clean up empty folders`

##### 4. Verify Structure (You should see: food101-train.tfrecord...):

`ls -R ~/tensorflow_datasets/food101/`

#### Step 4: Load in Notebook (Local):

Use download=False and point data_dir to your home directory to force tfds to use the local files.

`import tensorflow_datasets as tfds`
`import os`

Define path to local home directory:

`data_path = os.path.join(os.path.expanduser("~"), "tensorflow_datasets")`

Load data without downloading:

`(train_data, test_data), ds_info = tfds.load(name='food101', split=['train', 'validation'], shuffle_files=True, as_supervised=True, with_info=True, download=False, data_dir=data_path)`

In [17]:
# Load in The Data:

(train_data, test_data), ds_info = tfds.load(name= "food101", # target dataset to get from TFDS
                                            split= ["train","validation"], # what splits of data should we get? note: not all datasets have train, valid, test
                                            as_supervised= True, # download data in tuple format (sample, label), e.g. (image, label)
                                            shuffle_files= True, # shuffle files on download?
                                            with_info= True) # include dataset metadata? if so, tfds.load() returns tuple (data, ds_info)

After a few minutes of downloading, we've now got access to entire Food101 dataset (in tensor format) ready for modelling.

Now let's get a little information from our dataset, starting with the class names.

Getting class names from a TensorFlow Datasets dataset requires downloading the "`dataset_info`" variable (by using the `as_supervised=True` parameter in the `tfds.load()` method, **note:** this will only work for supervised datasets in TFDS).

We can access the class names of a particular dataset using the `dataset_info.features` attribute and accessing `names` attribute of the the `"label"` key.

In [19]:
# Features of 'food101':
ds_info.features

FeaturesDict({
    'image': Image(shape=(None, None, 3), dtype=uint8),
    'label': ClassLabel(shape=(), dtype=int64, num_classes=101),
})

In [20]:
type(ds_info)

tensorflow_datasets.core.dataset_info.DatasetInfo

In [21]:
# Getting Class Names:
class_names = ds_info.features['label'].names

In [23]:
class_names[:10]

['apple_pie',
 'baby_back_ribs',
 'baklava',
 'beef_carpaccio',
 'beef_tartare',
 'beet_salad',
 'beignets',
 'bibimbap',
 'bread_pudding',
 'breakfast_burrito']

In [24]:
len(class_names)

101

### Exploring the Food101 data from TensorFlow Datasets

Now we've downloaded the Food101 dataset from TensorFlow Datasets, let's explore it.

Let's find out a few details about our dataset:
* The shape of our input data (image tensors)
* The datatype of our input data
* What the labels of our input data look like (e.g. one-hot encoded versus label-encoded)
* Do the labels match up with the class names?

To do, let's take one sample off the training data (using the [`.take()` method](https://www.tensorflow.org/api_docs/python/tf/data/Dataset#take)) and explore it. 

In [25]:
# Taking one sample from Training Data:
train_one_sample = train_data.take(count= 1)

Because we used the `as_supervised=True` parameter in our `tfds.load()` method above, data samples come in the tuple format structure `(data, label)` or in our case `(image_tensor, label)`.

In [26]:
# What does One Sample from Training Data look like?
train_one_sample

<_TakeDataset element_spec=(TensorSpec(shape=(None, None, 3), dtype=tf.uint8, name=None), TensorSpec(shape=(), dtype=tf.int64, name=None))>

Let's loop through our single training sample and get some info from the `image_tensor` and `label`.

In [29]:
# Output info about One Training Sample:
for image, label in train_one_sample:
    print(f'''
    Image Shape: {image.shape},
    Image DataType: {image.dtype},
    Target Class from Food101 (Tensor form): {label},
    Class Name (Str form): {class_names[label.numpy()]}
    ''')


    Image Shape: (306, 512, 3),
    Image DataType: <dtype: 'uint8'>,
    Target Class from Food101 (Tensor form): 29,
    Class Name (Str form): cup_cakes
    


Because we set the `shuffle_files=True` parameter in our `tfds.load()` method above, running the cell above a few times will give a different result each time.

Checking these you might notice some of the images have different shapes, for example `(512, 342, 3)` and `(512, 512, 3)` (height, width, color_channels).

Let's see what one of the image tensors from TFDS's Food101 dataset looks like.

In [30]:
# What Does an Image Tensor from TFDS' FOOD101 looks like:
image

<tf.Tensor: shape=(306, 512, 3), dtype=uint8, numpy=
array([[[199, 100,  15],
        [192,  91,   9],
        [202,  95,  23],
        ...,
        [238, 196, 124],
        [237, 195, 123],
        [236, 194, 122]],

       [[210, 110,  25],
        [207, 105,  23],
        [194,  86,  13],
        ...,
        [238, 196, 124],
        [236, 194, 122],
        [235, 193, 121]],

       [[205, 103,  18],
        [213, 108,  25],
        [196,  86,  11],
        ...,
        [238, 195, 126],
        [237, 194, 125],
        [235, 192, 123]],

       ...,

       [[ 95,  28,  11],
        [ 94,  27,  10],
        [ 84,  17,   0],
        ...,
        [166,  32,   5],
        [166,  32,   3],
        [167,  34,   1]],

       [[ 87,  24,   7],
        [ 83,  18,   0],
        [ 87,  20,   1],
        ...,
        [155,  27,   0],
        [164,  39,   9],
        [156,  33,   0]],

       [[ 83,  21,   6],
        [ 79,  16,   0],
        [ 85,  19,   0],
        ...,
        [149,  25,   

In [31]:
import tensorflow as tf

# What are Minimum and Maximum values:
tf.reduce_min(image), tf.reduce_max(image)

(<tf.Tensor: shape=(), dtype=uint8, numpy=0>,
 <tf.Tensor: shape=(), dtype=uint8, numpy=255>)

Alright looks like our image tensors have values of between 0 & 255 (standard red, green, blue colour values) and the values are of data type `unit8`.

We might have to preprocess these before passing them to a neural network. But we'll handle this later.

In the meantime, let's see if we can plot an image sample.